# STTN-CP — Spatial-Temporal Transformer with Contrastive Pretraining

**Credit card fraud detection | model implementation**

This notebook exercises the same source implementation used by `src/train.py` and `src/evaluate.py`. It follows the STTN-CP modeling formulation of spatial attention over transaction features, temporal attention over transaction windows, residual connections, contrastive representation learning, and supervised classification.

The notebook is intentionally separated from the archived `results/original_run/` record. A dataset-backed run here uses the current repository protocol; the archived original experiment is preserved separately.

## 1. Imports and configuration

The notebook imports the implementation directly from `src/` so that architecture behavior, data handling, loss functions, and evaluation remain consistent with the repository tests and command-line workflow.

In [ ]:
from pathlib import Path
import json
import sys

import numpy as np
import pandas as pd
import torch
from torch import nn

ROOT = Path.cwd()
if not (ROOT / 'src').exists() and (ROOT.parent / 'src').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.architecture import (
    STTNCP,
    combined_loss,
    count_trainable_parameters,
    infonce_loss,
)
from src.data import (
    build_labeled_windows,
    class_weights,
    chronological_split,
    load_creditcard,
    make_loader,
    scale_splits,
)
from src.train import augment, predict_and_score

SEED = 42
SEQ_LEN = 8
EMBED_DIM = 128
N_BLOCKS = 3
N_HEADS = 4
MLP_DIM = 256
PROJ_DIM = 64
BATCH_SIZE = 128
EPOCHS = 50
LR = 1e-3
TEMPERATURE = 0.07
LAMBDA_CONTRASTIVE = 0.5
PATIENCE = 10
NOISE_STD = 0.02
DATA_PATH = ROOT / 'data' / 'creditcard.csv'
RUN_TRAINING = False  # Set True for a dataset-backed training run.

np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f'Device: {DEVICE}')
print(f'Dataset present: {DATA_PATH.exists()} -> {DATA_PATH}')

## 2. Dataset and current repository protocol

The maintained implementation loads the Kaggle credit-card dataset, sorts transactions by `Time`, creates chronological 70% training / 10% validation / 20% test partitions, fits `MinMaxScaler` on the training partition only, and constructs sequence windows independently within each partition.

Each 8-transaction window receives the class label of its final transaction. This prevents a sequence from crossing train/validation/test boundaries.

In [ ]:
if DATA_PATH.exists():
    df = load_creditcard(DATA_PATH)
    print('Dataset shape:', df.shape)
    print('Class counts:')
    print(df['Class'].value_counts().sort_index())

    splits = chronological_split(
        df,
        train_fraction=0.70,
        validation_fraction=0.10,
    )
    print('\nPartitions:', len(splits.train), len(splits.validation), len(splits.test))

    (X_train, y_train), (X_val, y_val), (X_test, y_test), scaler, features = scale_splits(
        splits.train,
        splits.validation,
        splits.test,
    )
    print('Feature count:', len(features))
    print('First features:', features[:5])
    print('Last features:', features[-3:])

    X_train_w, y_train_w = build_labeled_windows(X_train, y_train, SEQ_LEN)
    X_val_w, y_val_w = build_labeled_windows(X_val, y_val, SEQ_LEN)
    X_test_w, y_test_w = build_labeled_windows(X_test, y_test, SEQ_LEN)
    print('Window shapes:', X_train_w.shape, X_val_w.shape, X_test_w.shape)
else:
    print('creditcard.csv is not present. Dataset-backed cells are skipped; the software smoke test below remains runnable.')


## 3. STTN-CP architecture

The implementation follows the central STTN-CP flow:

1. project each transaction window into an embedding space;
2. apply spatial self-attention across the original transaction features at each timestep;
3. add the spatial representation through a residual connection;
4. apply temporal self-attention across the transaction sequence;
5. add the temporal representation through a residual connection;
6. stack the spatial-temporal blocks and aggregate the resulting sequence representation;
7. use a contrastive projection head for InfoNCE and a classification head for fraud logits.

This matches the repository architecture tests, which explicitly check the feature-token direction, complete `(B, T, D)` input, and multiple stacked ST blocks.

In [ ]:
model = STTNCP(
    input_dim=30,
    seq_len=SEQ_LEN,
    embed_dim=EMBED_DIM,
    n_blocks=N_BLOCKS,
    n_heads=N_HEADS,
    mlp_dim=MLP_DIM,
    proj_dim=PROJ_DIM,
).to(DEVICE)

print(model)
print(f'\nTrainable parameters: {count_trainable_parameters(model):,}')

with torch.no_grad():
    sample = torch.randn(4, SEQ_LEN, 30, device=DEVICE)
    backbone = model.forward_backbone(sample)
    logits = model(sample)
    projection = model.project(backbone)

print('Input shape:', tuple(sample.shape))
print('Backbone shape:', tuple(backbone.shape))
print('Projection shape:', tuple(projection.shape))
print('Classifier logits shape:', tuple(logits.shape))

## 4. Contrastive objective and classification objective

The contrastive component uses symmetric InfoNCE with cosine-normalized representations and temperature `τ = 0.07`.

The combined optimization objective is:

$$
L_{\mathrm{total}}
=L_{\mathrm{classification}}+\lambda L_{\mathrm{contrastive}}
$$

with `λ = 0.5` in the maintained configuration.

The current classifier produces two class logits and is trained with weighted cross-entropy. This is the exact behavior of `src/architecture.py` and `src/train.py`.

In [ ]:
z1 = torch.randn(8, PROJ_DIM)
z2 = torch.randn(8, PROJ_DIM)
contrastive_value = infonce_loss(z1, z2, temperature=TEMPERATURE)

example_logits = torch.randn(8, 2)
example_labels = torch.randint(0, 2, (8,))
criterion = nn.CrossEntropyLoss()
total_value, class_value, contrast_value = combined_loss(
    example_logits,
    example_labels,
    z1,
    z2,
    criterion,
    lambda_contrastive=LAMBDA_CONTRASTIVE,
    temperature=TEMPERATURE,
)

print('Standalone InfoNCE loss:', float(contrastive_value))
print('Classification loss:', float(class_value))
print('Contrastive loss:', float(contrast_value))
print('Combined loss:', float(total_value))
print('Combined loss check:', np.isclose(
    float(total_value),
    float(class_value) + LAMBDA_CONTRASTIVE * float(contrast_value),
))

## 5. Optional dataset-backed training run

Set `RUN_TRAINING = True` above to execute the maintained training workflow on `data/creditcard.csv`.

This uses the same defaults as `src/train.py`: sequence length 8, 3 ST blocks, embedding width 128, 4 attention heads, batch size 128, Adam with learning rate 0.001, InfoNCE temperature 0.07, `λ = 0.5`, and early stopping with patience 10.

For a normal repository run, the command-line interface is the preferred reproducible entry point.

In [ ]:
if RUN_TRAINING:
    if not DATA_PATH.exists():
        raise FileNotFoundError(f'Missing dataset: {DATA_PATH}')

    from argparse import Namespace
    from src.train import train

    args = Namespace(
        data_path=str(DATA_PATH),
        output_dir=str(ROOT / 'outputs'),
        seed=SEED,
        seq_len=SEQ_LEN,
        embed_dim=EMBED_DIM,
        n_blocks=N_BLOCKS,
        n_heads=N_HEADS,
        mlp_dim=MLP_DIM,
        proj_dim=PROJ_DIM,
        batch_size=BATCH_SIZE,
        epochs=EPOCHS,
        lr=LR,
        temperature=TEMPERATURE,
        lambda_contrastive=LAMBDA_CONTRASTIVE,
        patience=PATIENCE,
        noise_std=NOISE_STD,
    )

    training_summary = train(args)
    print(json.dumps(training_summary['test_metrics'], indent=2))
else:
    print('RUN_TRAINING is False; no dataset-backed training run was started.')

## 6. Software smoke test

The following test uses a deterministic synthetic dataset only to exercise the complete model path. Its metrics are not research benchmark results.

In [ ]:
def synthetic_windows(n=256, seq_len=8, input_dim=30, seed=42):
    rng = np.random.default_rng(seed)
    X = rng.normal(size=(n + seq_len - 1, input_dim)).astype(np.float32)
    y = (X[:, 0] + 0.2 * X[:, 1] > 0).astype(np.int64)
    return build_labeled_windows(X, y, seq_len)

sx, sy = synthetic_windows()
smoke_model = STTNCP(
    input_dim=30,
    seq_len=SEQ_LEN,
    embed_dim=32,
    n_blocks=1,
    n_heads=4,
    mlp_dim=64,
    proj_dim=16,
).eval()

with torch.no_grad():
    smoke_input = torch.from_numpy(sx[:16])
    smoke_logits = smoke_model(smoke_input)
    smoke_embedding = smoke_model.forward_backbone(smoke_input)
    smoke_projection = smoke_model.project(smoke_embedding)
    smoke_loss = infonce_loss(smoke_projection[:8], smoke_projection[8:16])

print('Synthetic windows:', sx.shape)
print('Classifier output:', tuple(smoke_logits.shape))
print('Backbone output:', tuple(smoke_embedding.shape))
print('Projection output:', tuple(smoke_projection.shape))
print('InfoNCE loss finite:', bool(torch.isfinite(smoke_loss)))

## 7. Evaluation on a saved checkpoint

The repository evaluation script reconstructs the model from the saved checkpoint, restores the training scaler, applies the current chronological test split, creates partition-local windows, and writes the evaluation metrics to JSON.

The same behavior is exposed through the command line:

In [ ]:
print('python -m src.evaluate \\')
print('  --data-path data/creditcard.csv \\')
print('  --checkpoint outputs/best_model.pt \\')
print('  --output outputs/test_metrics.json')

## 8. Research result reference

For research context, the reported STTN-CP evaluation reports the following test-set values:

| Metric | Reported value |
|---|---:|
| Accuracy | 99.12% |
| Precision | 99.00% |
| Recall | 98.86% |
| F1 | 98.92% |
| Specificity | 97.96% |

These values are shown here as a research reference only. They are not generated by the synthetic smoke test and are not substituted into the current repository output.

## 9. Running the maintained implementation

Install dependencies:

```bash
pip install -r requirements.txt
```

Train:

```bash
python -m src.train \
  --data-path data/creditcard.csv \
  --output-dir outputs
```

Evaluate:

```bash
python -m src.evaluate \
  --data-path data/creditcard.csv \
  --checkpoint outputs/best_model.pt \
  --output outputs/test_metrics.json
```

Run tests:

```bash
python -m pytest -q
```
